# Késni fog-e a járat? — 1. rész: adat-előkészítés és EDA

**Adat:** 2015 Flight Delays and Cancellations (Kaggle `usdot/flight-delays`), a **Delta** (`DL`) járatai. A projekt terve: `projektterv.md`.

**Szabály:** az EDA csak a **tanító időszakon** (január–szeptember) készül; az október a rés, a november–december a zárolt teszt, ide nem nézünk.
Az EDA-ban a késés okait és a tényleges időpontokat is használhatjuk *leírásra*, de ezek a modellben szivárgás lennének (lásd a terv 3. pontját).

**Futtatás:** a Kaggle-ről letöltött `flights.csv`, `airports.csv` és `airlines.csv` a `data/` mappában legyen. Az első futás kivágja a Delta-járatokat
és elmenti őket (`data/delta_2015.pkl`); utána a notebook ebből a gyorsítótárból indul.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import PercentFormatter

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
pd.set_option("display.width", 180)
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "yellow": "#eda100", "magenta": "#e87ba4",
     "grey": "#a9a8a2", "ink": "#0b0b0b", "ink2": "#52514e"}
SEQ = LinearSegmentedColormap.from_list("késés", ["#fde3d6", "#eb6834", "#7a2a0c"])     # világos = kevés késés, sötét = sok
plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "font.size": 10,
                     "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 10,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#c3c2b7",
                     "axes.labelcolor": C["ink2"], "xtick.color": C["ink2"], "ytick.color": C["ink2"],
                     "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6, "legend.frameon": False})
pct = PercentFormatter(1.0, decimals=0)

DATA = Path(os.environ.get("FP_DATA", "data"))
CACHE, TRAFFIC = DATA / "delta_2015.pkl", DATA / "traffic_2015.pkl"

## 1. Betöltés és a Delta-járatok kivágása

Az 5,8 millió soros fájlt **darabonként** olvassuk, csak a szükséges oszlopokkal. Közben két dolgot gyűjtünk:
a Delta-járatokat, és **minden** légitársaság óránkénti indulásainak számát repülőterenként (ez a zsúfoltság-jellemzőhöz kell).

In [ ]:
COLS = ["MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
        "SCHEDULED_DEPARTURE", "DEPARTURE_TIME", "DEPARTURE_DELAY", "SCHEDULED_TIME", "DISTANCE", "SCHEDULED_ARRIVAL",
        "ARRIVAL_TIME", "ARRIVAL_DELAY", "DIVERTED", "CANCELLED", "CANCELLATION_REASON",
        "AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]
TXT = {c: str for c in ["AIRLINE", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "CANCELLATION_REASON"]}

if CACHE.exists():
    dl, traffic = pd.read_pickle(CACHE), pd.read_pickle(TRAFFIC)
    n_all = int(traffic.sum())
else:
    parts, traf, n_all = [], [], 0
    for ch in pd.read_csv(DATA / "flights.csv", usecols=COLS, dtype=TXT, chunksize=500_000):
        n_all += len(ch)
        ch["dep_hour"] = (ch.SCHEDULED_DEPARTURE // 100).clip(upper=23)
        traf.append(ch.groupby(["ORIGIN_AIRPORT", "MONTH", "DAY", "dep_hour"]).size())
        parts.append(ch[ch.AIRLINE == "DL"])
    dl = pd.concat(parts, ignore_index=True)
    traffic = pd.concat(traf).groupby(level=[0, 1, 2, 3]).sum().rename("traffic")
    dl.to_pickle(CACHE); traffic.to_pickle(TRAFFIC)

print(f"összes járat: {n_all:,} | Delta: {len(dl):,} ({len(dl) / n_all:.1%}) | gépek: {dl.TAIL_NUMBER.nunique():,} | "
      f"repülőterek: {pd.concat([dl.ORIGIN_AIRPORT, dl.DESTINATION_AIRPORT]).nunique():,}")

## 2. Alap-előkészítés

Dátum és tervezett indulás (helyi idő; a `2400` éjfélt jelöl), a célváltozó (érkezési késés > 15 perc), és a split-időszakok.

In [ ]:
hhmm = dl.SCHEDULED_DEPARTURE.astype(int).clip(upper=2359)
dl["date"] = pd.to_datetime(dict(year=2015, month=dl.MONTH, day=dl.DAY))
dl["sched_dep"] = dl.date + pd.to_timedelta(hhmm // 100, unit="h") + pd.to_timedelta(hhmm % 100, unit="m")
dl["dep_hour"] = (hhmm // 100).astype(int)
dl["delayed"] = (dl.ARRIVAL_DELAY > 15).astype(float).where(dl.ARRIVAL_DELAY.notna())     # törölt / elterelt: NaN
dl["period"] = np.select([dl.MONTH <= 9, dl.MONTH == 10], ["tanító", "rés"], "teszt")
print(dl.period.value_counts().reindex(["tanító", "rés", "teszt"]).to_string())

tr = dl[dl.period == "tanító"].copy()          # MINDEN alábbi EDA csak ezen fut
done = tr[tr.ARRIVAL_DELAY.notna()]             # ténylegesen megérkezett járatok

## 3. Adatminőség

**Kérdés:** mi hiányzik vagy furcsa? **Döntés:** tisztítási szabályok (a modell előtt, a teszten mechanikusan ugyanúgy).

In [ ]:
numeric_code = dl.ORIGIN_AIRPORT.str.fullmatch(r"\d+")
quality = pd.DataFrame({
    "számkódú repülőtér (havonta)": numeric_code.groupby(dl.MONTH).mean(),
    "hiányzó lajstromjel": dl.TAIL_NUMBER.isna().groupby(dl.MONTH).mean(),
    "törölt": dl.CANCELLED.groupby(dl.MONTH).mean(),
    "elterelt": dl.DIVERTED.groupby(dl.MONTH).mean(),
})
print(f"tervezett indulás 2400 (éjfél): {(dl.SCHEDULED_DEPARTURE == 2400).sum()} sor")
quality.style.format("{:.2%}")

*Értelmezés: a valódi adaton való futtatás után.*

## 4. A célváltozó

**Kérdés:** hogyan oszlik el az érkezési késés, és hány járat késik 15 percnél többet? **Döntés:** a cél definíciója és a fő mutató (osztályarány → PR-AUC).

In [ ]:
rate = done.delayed.mean()
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.hist(done.ARRIVAL_DELAY.clip(-60, 240), bins=np.arange(-60, 245, 5), color=C["blue"], edgecolor="#fcfcfb", linewidth=0.4)
ax.axvline(15, color=C["orange"], lw=2)
ax.text(20, ax.get_ylim()[1] * 0.85, f"15 percnél többet késik: {rate:.1%}", color=C["orange"])
ax.set(title="A legtöbb járat időben érkezik, de a késések hosszú farkat húznak",
       xlabel="érkezési késés (perc; −60 és 240 közé vágva, negatív = korai)", ylabel="járat")
plt.tight_layout(); plt.show()
q = done.ARRIVAL_DELAY.quantile([0.5, 0.9, 0.95, 0.99])
print(f"késési arány (> 15 perc): {rate:.1%} | medián {q[0.5]:.0f} perc | 90%: {q[0.9]:.0f} | 95%: {q[0.95]:.0f} | 99%: {q[0.99]:.0f} perc")

*Értelmezés: a valódi adaton való futtatás után.*

## 5. Mi okozza a késéseket?

**Kérdés:** a késett percek hogyan oszlanak meg az öt ok között, és változik-e ez havonta? **Döntés:** mennyit adhat a gép előző járata és az időjárás.
(Ezek az oszlopok csak utólag ismertek: leírásra jók, jellemzőnek nem.)

In [ ]:
CAUSES = {"LATE_AIRCRAFT_DELAY": "késve érkező gép", "AIRLINE_DELAY": "légitársaság", "AIR_SYSTEM_DELAY": "légtér-rendszer",
          "WEATHER_DELAY": "szélsőséges időjárás", "SECURITY_DELAY": "biztonság"}
mins = done[list(CAUSES)].rename(columns=CAUSES)
share = mins.sum() / mins.sum().sum()
monthly = mins.groupby(done.MONTH).sum()
monthly = monthly.div(monthly.sum(axis=1), axis=0)

fig, ax = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [1, 1.3]})
s = share.sort_values()
ax[0].barh(s.index, s.values, color=[C["orange"] if v == s.max() else C["blue"] for v in s.values], height=0.6)
for i, v in enumerate(s.values):
    ax[0].text(v + 0.01, i, f"{v:.0%}", va="center")
ax[0].xaxis.set_major_formatter(pct); ax[0].grid(axis="y", visible=False)
ax[0].set(title="A késett percek megoszlása okonként", xlabel="a késett percek aránya (jan.–szept.)")
for col, name in zip([C["blue"], C["orange"], C["aqua"], C["yellow"], C["magenta"]], CAUSES.values()):
    ax[1].plot(monthly.index, monthly[name], "o-", color=col, lw=2, ms=5, label=name)
ax[1].yaxis.set_major_formatter(pct); ax[1].set_xticks(range(1, 10))
ax[1].set(title="Havonta", xlabel="hónap", ylabel="a késett percek aránya")
ax[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), fontsize=9, ncol=5)
plt.tight_layout(); plt.show()
print((share * 100).round(1).astype(str).add(" %").to_string())

*Értelmezés: a valódi adaton való futtatás után.*

## 6. Mikor késnek a járatok?

**Kérdés:** nő-e a késés a nap folyamán, és melyik napokon nagyobb? **Döntés:** az indulás órája és a hét napja jellemző.

In [ ]:
by_hour = done.groupby("dep_hour").delayed.agg(["mean", "size"])
by_hour = by_hour[by_hour["size"] >= 100]
by_dow = done.groupby("DAY_OF_WEEK").delayed.mean()
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8), gridspec_kw={"width_ratios": [2, 1]})
ax[0].bar(by_hour.index, by_hour["mean"], color=C["blue"], width=0.75)
peak = by_hour["mean"].idxmax()
ax[0].bar(peak, by_hour.loc[peak, "mean"], color=C["orange"], width=0.75)
ax[0].yaxis.set_major_formatter(pct); ax[0].grid(axis="x", visible=False)
ax[0].set(title="A késés a nap folyamán felhalmozódik", xlabel="tervezett indulás órája (helyi idő)", ylabel="késési arány (> 15 perc)")
ax[1].bar(["H", "K", "Sze", "Cs", "P", "Szo", "V"], by_dow.values, color=C["blue"], width=0.6)
ax[1].yaxis.set_major_formatter(pct); ax[1].grid(axis="x", visible=False)
ax[1].set(title="A hét napjai szerint", ylabel="késési arány")
plt.tight_layout(); plt.show()
print(f"legkisebb: {by_hour['mean'].idxmin()} óra ({by_hour['mean'].min():.1%}) | legnagyobb: {peak} óra ({by_hour['mean'].max():.1%})")

*Értelmezés: a valódi adaton való futtatás után.*

**Kérdés:** van-e szezonalitás, és kiugranak-e az ünnepek vagy a viharos napok? **Döntés:** az időrendi split, és egy ünnep-jellemző.

In [ ]:
daily = done.groupby("date").delayed.mean()
HOLIDAYS = {"2015-01-01": "újév", "2015-05-25": "Memorial Day", "2015-07-04": "július 4.", "2015-09-07": "Labor Day"}
fig, ax = plt.subplots(figsize=(14, 3.8))
ax.plot(daily.index, daily.values, color=C["grey"], lw=1, label="napi")
ax.plot(daily.index, daily.rolling(7, center=True).mean(), color=C["blue"], lw=2.2, label="7 napos átlag")
for d, name in HOLIDAYS.items():
    if pd.Timestamp(d) in daily.index:
        ax.axvline(pd.Timestamp(d), color=C["orange"], lw=1)
        ax.text(pd.Timestamp(d), ax.get_ylim()[1] * 0.95, " " + name, color=C["orange"], fontsize=9, va="top")
worst = daily.nlargest(3)
ax.scatter(worst.index, worst.values, color=C["orange"], zorder=3, s=30)
ax.yaxis.set_major_formatter(pct); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2)
ax.set(title="Napi késési arány: a rossz napok kiugranak (narancs pont: a 3 legrosszabb nap)", ylabel="késési arány (> 15 perc)")
plt.tight_layout(); plt.show()
print("a 3 legrosszabb nap:", {d.date().isoformat(): f"{v:.0%}" for d, v in worst.items()})
print("havi késési arány:", {m: f"{v:.1%}" for m, v in done.groupby("MONTH").delayed.mean().items()})

*Értelmezés: a valódi adaton való futtatás után.*

## 7. Hol késnek a járatok?

**Kérdés:** mely repülőterekről induló Delta-járatok késnek a legtöbbet? **Döntés:** reptér-jellemző; a történeti késési arány csak foldon belül (célkódolás).

In [ ]:
ap = pd.read_csv(DATA / "airports.csv")
by_ap = done.groupby("ORIGIN_AIRPORT").delayed.agg(arány="mean", járat="size").reset_index()
by_ap = by_ap.merge(ap, left_on="ORIGIN_AIRPORT", right_on="IATA_CODE")
by_ap = by_ap[(by_ap.járat >= 300) & by_ap.LONGITUDE.between(-130, -60) & by_ap.LATITUDE.between(23, 50)]   # a szárazföldi USA
fig, ax = plt.subplots(figsize=(13, 6))
sc = ax.scatter(by_ap.LONGITUDE, by_ap.LATITUDE, s=by_ap.járat / by_ap.járat.max() * 900 + 15, c=by_ap.arány, cmap=SEQ,
                alpha=0.85, edgecolors="#fcfcfb", linewidths=0.8)
for _, r in by_ap.nlargest(6, "járat").iterrows():
    ax.annotate(r.ORIGIN_AIRPORT, (r.LONGITUDE, r.LATITUDE), textcoords="offset points", xytext=(6, 6), fontsize=9, color=C["ink"])
cb = plt.colorbar(sc, ax=ax, format=PercentFormatter(1.0, decimals=1)); cb.set_label("késési arány"); cb.outline.set_visible(False)
ax.set(title="Induló repülőterek: a pont mérete a Delta-forgalom, a színe a késési arány", xlabel="hosszúság", ylabel="szélesség"); ax.grid(False)
plt.tight_layout(); plt.show()
by_ap.sort_values("járat", ascending=False)[["ORIGIN_AIRPORT", "CITY", "járat", "arány"]].head(10)

*Értelmezés: a valódi adaton való futtatás után.*

## 8. Zsúfoltság

**Kérdés:** nagyobb-e a késés, ha az adott órában sok gép indul a repülőtérről (az összes légitársasággal együtt)? **Döntés:** forgalom-jellemző (a menetrendből előre ismert).

In [ ]:
done = done.join(traffic, on=["ORIGIN_AIRPORT", "MONTH", "DAY", "dep_hour"])
bins = pd.qcut(done.traffic, 10, duplicates="drop")
by_traf = done.groupby(bins, observed=True).agg(forgalom=("traffic", "median"), arány=("delayed", "mean"))
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(by_traf.forgalom, by_traf.arány, "o-", color=C["blue"], lw=2, ms=7)
ax.yaxis.set_major_formatter(pct)
ax.set(title="Zsúfoltság és késés", xlabel="tervezett indulások az adott órában a repülőtéren (decilisek mediánja)", ylabel="késési arány")
plt.tight_layout(); plt.show()
by_traf

*Értelmezés: a valódi adaton való futtatás után.*

## 9. Továbbgyűrűzik-e a késés?

**Kérdés:** ha a gép előző járata késett, a következő is késik? És számít-e a fordulóidő (az előző járat tervezett érkezése és ennek tervezett indulása között)?
**Döntés:** a projekt legerősebb jellemzője. *(Itt leíró elemzés: az előző járat tényleges késését nézzük. A modellben csak az lesz jellemző, ami az indulás előtt 2 órával már ismert.)*

In [ ]:
seq = tr[tr.TAIL_NUMBER.notna()].sort_values(["TAIL_NUMBER", "sched_dep"]).copy()
g = seq.groupby(["TAIL_NUMBER", "date"])
seq["prev_arr_delay"] = g.ARRIVAL_DELAY.shift(1)
arr_hhmm = g.SCHEDULED_ARRIVAL.shift(1)
prev_arr = seq.date + pd.to_timedelta(arr_hhmm // 100, unit="h") + pd.to_timedelta(arr_hhmm % 100, unit="m")
seq["turn_min"] = (seq.sched_dep - prev_arr).dt.total_seconds() / 60          # tervezett fordulóidő (perc)
seq = seq[seq.prev_arr_delay.notna() & seq.delayed.notna() & seq.turn_min.between(10, 600)]

prev_bins = pd.cut(seq.prev_arr_delay, [-np.inf, 0, 15, 30, 60, 120, np.inf], labels=["korai", "0–15", "15–30", "30–60", "60–120", "120+"])
by_prev = seq.groupby(prev_bins, observed=True).delayed.agg(["mean", "size"])
late_prev = seq[seq.prev_arr_delay > 15]
turn_bins = pd.cut(late_prev.turn_min, [10, 40, 60, 90, 120, 180, 600], labels=["10–40", "40–60", "60–90", "90–120", "120–180", "180+"])
by_turn = late_prev.groupby(turn_bins, observed=True).delayed.agg(["mean", "size"])

fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].bar(by_prev.index.astype(str), by_prev["mean"], width=0.65,
          color=[C["blue"] if lab in ("korai", "0–15") else C["orange"] for lab in by_prev.index.astype(str)])
ax[0].axhline(rate, color=C["ink2"], lw=1)
ax[0].text(0.99, rate, "átlag ", transform=ax[0].get_yaxis_transform(), ha="right", va="bottom", color=C["ink2"])
ax[0].yaxis.set_major_formatter(pct); ax[0].grid(axis="x", visible=False)
ax[0].set(title="Ha a gép késve érkezett, a következő járata is késik", xlabel="a gép előző járatának érkezési késése (perc)", ylabel="késési arány")
ax[1].plot(by_turn.index.astype(str), by_turn["mean"], "o-", color=C["orange"], lw=2, ms=7)
ax[1].yaxis.set_major_formatter(pct)
ax[1].set(title="Hosszabb fordulóidő: kevesebb átvitt késés", xlabel="tervezett fordulóidő (perc), ha az előző járat késett", ylabel="késési arány")
plt.tight_layout(); plt.show()
print(f"korreláció (az előző járat és a mostani érkezési késése, ugyanazon a napon): {seq.prev_arr_delay.corr(seq.ARRIVAL_DELAY):.2f} | "
      f"a gép napi első járata nélkül: {len(seq):,} sor")
by_prev

*Értelmezés: a valódi adaton való futtatás után.*

## 10. Törlések és elterelések

**Kérdés:** milyen gyakori, és miért? **Döntés:** a fő elemzésből kimaradnak (nincs érkezési késésük), de a korlátoknál említjük.
Ok-kódok: A = légitársaság, B = időjárás, C = légtér-rendszer, D = biztonság.

In [ ]:
canc = tr[tr.CANCELLED == 1]
by_reason = canc.groupby(["MONTH", "CANCELLATION_REASON"]).size().unstack(fill_value=0)
by_reason = by_reason.reindex(columns=["A", "B", "C", "D"], fill_value=0).div(tr.groupby("MONTH").size(), axis=0)
fig, ax = plt.subplots(figsize=(10, 3.8))
bottom = np.zeros(len(by_reason))
for code, name, col in [("A", "légitársaság", C["blue"]), ("B", "időjárás", C["orange"]), ("C", "légtér-rendszer", C["aqua"]), ("D", "biztonság", C["yellow"])]:
    ax.bar(by_reason.index, by_reason[code], bottom=bottom, color=col, label=name, width=0.65, edgecolor="#fcfcfb", linewidth=1)
    bottom += by_reason[code].values
ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1)); ax.set_xticks(range(1, 10)); ax.grid(axis="x", visible=False)
ax.legend(loc="upper right", ncol=4, fontsize=9)
ax.set(title="Törölt járatok aránya havonta, ok szerint", xlabel="hónap", ylabel="a havi járatok aránya")
plt.tight_layout(); plt.show()
print(f"törölt: {tr.CANCELLED.mean():.2%} | elterelt: {tr.DIVERTED.mean():.2%} (a tanító időszakon)")

*Értelmezés: a valódi adaton való futtatás után.*

## 11. Összefoglaló: amit az EDA alapján a modellezésben csinálunk

*A valódi adaton való futtatás után töltöm ki: cél és osztályarány, tisztítási szabályok, a jellemzők listája, és amit a 2. rész (időjárás) és a 3. rész (modellezés) megkap.*